# Warm Hybrid Ablation
Compares three warm-path fusion strategies on the same 1000-user, 80/20 held-out protocol:
- **Adaptive** (current default): alpha = sigmoid((log1p(n)−2)/1.5), min-max normalised blend
- **Fixed α=0.5**: equal CF/content weight, min-max normalised blend
- **RRF**: Reciprocal Rank Fusion, no normalisation, no alpha (Cormack et al. 2009)

Evaluated under both standard and long-tail (top-100 excluded) protocols.
CF is included as a reference ceiling.

In [ ]:
import os, sys
os.environ['OPENBLAS_NUM_THREADS'] = '1'
from pathlib import Path
sys.path.insert(0, str(Path('..').resolve()))

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from scipy.sparse import csr_matrix

from src.data import load_msd_metadata, load_spotify_tracks, match_datasets, build_metadata_catalog
from src.models import CFModel, ContentModel, PopularityBaseline
from src.hybrid import HybridRecommender
from src.evaluation import evaluate_model, bootstrap_ci, paired_bootstrap_test

MODELS_DIR  = Path('../models')
RESULTS_DIR = Path('../results')
K_VALUES    = [5, 10, 20]


In [ ]:
cf      = CFModel.load(MODELS_DIR)
cm      = ContentModel.load(MODELS_DIR)
triplets = joblib.load(RESULTS_DIR / 'triplets_cache.pkl')
pop     = PopularityBaseline().fit(triplets)

msd_meta = load_msd_metadata()
spotify  = load_spotify_tracks()
matched  = match_datasets(spotify, msd_meta)
meta_cat = build_metadata_catalog(matched)

# Three hybrid variants
hybrid_adaptive = HybridRecommender(cf, cm, meta_cat, alpha_strategy='adaptive')
hybrid_fixed    = HybridRecommender(cf, cm, meta_cat, alpha_strategy='fixed', alpha=0.5)
hybrid_rrf      = HybridRecommender(cf, cm, meta_cat, alpha_strategy='rrf')
cf_song_to_idx  = {sid: idx for idx, sid in cf._idx_to_song.items()}

split                  = joblib.load(RESULTS_DIR / 'eval_split.pkl')
sampled_users          = split['sampled_users']
test_data              = split['test_data']
user_train_songs       = split['user_train_songs']
user_most_played_in_ct = split['user_most_played_in_ct']
print(f'Loaded {len(sampled_users)} users, 3 hybrid variants ready')


In [ ]:
# Long-tail ground truth (same as warm_longtail.ipynb)
top100_global = set(
    triplets.groupby('sid')['count'].sum()
    .sort_values(ascending=False).head(100)
    .index.astype(str)
)
test_data_lt = {uid: (held - top100_global)
                for uid, held in test_data.items()
                if held - top100_global}
print(f'Long-tail: {len(test_data_lt)} users (dropped {len(test_data)-len(test_data_lt)})')


In [ ]:
def make_cf_fn():
    def cf_fn(uid, k):
        uidx      = cf._user_to_idx[uid]
        full      = cf._user_item[uidx].toarray().flatten()
        mask      = np.zeros(full.shape, dtype=bool)
        for sid in user_train_songs[uid]:
            if sid in cf_song_to_idx:
                mask[cf_song_to_idx[sid]] = True
        train_row = csr_matrix(full * mask)
        idxs, _   = cf._model.recommend(uidx, train_row, N=k,
                                         filter_already_liked_items=True)
        return [cf._idx_to_song[int(i)] for i in idxs]
    return cf_fn

def make_hybrid_fn(h):
    def hybrid_fn(uid, k):
        return h.recommend(
            user_id=uid, k=k,
            known_song_ids=set(user_train_songs[uid]),
        )['song_id'].tolist()
    return hybrid_fn

model_fns = {
    'CF':              make_cf_fn(),
    'Hybrid-Adaptive': make_hybrid_fn(hybrid_adaptive),
    'Hybrid-Fixed0.5': make_hybrid_fn(hybrid_fixed),
    'Hybrid-RRF':      make_hybrid_fn(hybrid_rrf),
}


In [ ]:
_ABL_STD = RESULTS_DIR / 'warm_hybrid_ablation_per_user.csv'
_ABL_LT  = RESULTS_DIR / 'warm_hybrid_ablation_per_user_lt.csv'

for cache_path, tdata, label in [
    (_ABL_STD, test_data,    'standard'),
    (_ABL_LT,  test_data_lt, 'long-tail'),
]:
    if cache_path.exists():
        print(f'Cache hit: {cache_path.name}')
        continue
    frames = []
    for name, fn in model_fns.items():
        print(f'  {label} {name}...', end=' ', flush=True)
        df = evaluate_model(fn, tdata, k_values=K_VALUES)
        df['model'] = name
        frames.append(df)
        print('done')
    pd.concat(frames, ignore_index=True).to_csv(cache_path, index=False)
    print(f'Saved {cache_path.name}')

per_user_std = pd.read_csv(_ABL_STD)
per_user_lt  = pd.read_csv(_ABL_LT)
print(f'Standard: {len(per_user_std):,} rows  |  Long-tail: {len(per_user_lt):,} rows')


In [ ]:
MODEL_ORDER  = ['CF', 'Hybrid-Adaptive', 'Hybrid-Fixed0.5', 'Hybrid-RRF']
METRIC_ORDER = ['ndcg', 'hit_rate', 'recall']

def make_agg(per_user_df, n_users):
    rows = []
    for model in MODEL_ORDER:
        for k in K_VALUES:
            for metric in METRIC_ORDER:
                scores = per_user_df[
                    (per_user_df['model']==model) &
                    (per_user_df['k']==k) &
                    (per_user_df['metric']==metric)
                ]['value'].values
                pt, lo, hi = bootstrap_ci(scores)
                rows.append({'model':model,'k':k,'metric':metric,
                             'mean':pt,'ci_low':lo,'ci_high':hi})
    return pd.DataFrame(rows)

agg_std = make_agg(per_user_std, len(test_data))
agg_lt  = make_agg(per_user_lt,  len(test_data_lt))

# Merge and save
agg_std['protocol'] = 'standard'
agg_lt['protocol']  = 'long-tail'
ablation = pd.concat([agg_std, agg_lt], ignore_index=True)
ablation.to_csv(RESULTS_DIR / 'warm_hybrid_ablation.csv', index=False)
print('Saved warm_hybrid_ablation.csv\n')

for protocol, agg in [('STANDARD', agg_std), ('LONG-TAIL', agg_lt)]:
    n = len(test_data) if protocol=='STANDARD' else len(test_data_lt)
    print(f'=== {protocol} @ k=10 (n={n}) ===')
    print(f'{"Model":<20} {"NDCG@10":<24} {"HitRate@10":<24} {"Recall@10":<24}')
    print('-'*92)
    for model in MODEL_ORDER:
        parts = [f'{model:<20}']
        for metric in ['ndcg','hit_rate','recall']:
            r = agg[(agg['model']==model)&(agg['k']==10)&(agg['metric']==metric)].iloc[0]
            parts.append(f'{r["mean"]:.4f} [{r["ci_low"]:.4f},{r["ci_high"]:.4f}]')
        print('  '.join(f'{p:<24}' for p in parts))
    print()


In [ ]:
# Hypothesis tests: each hybrid variant vs CF (NDCG@10, both protocols)
def get_scores(df, model):
    return (df[(df['model']==model)&(df['k']==10)&(df['metric']=='ndcg')]
            .sort_values('user_id')['value'].values)

ht_rows = []
print(f'{"Protocol":<12} {"Comparison":<32} {"delta":>10} {"p":>8} {"d":>8}  Effect')
print('-'*78)

def cohens_d(a, b):
    diff = a - b
    return float(diff.mean() / diff.std(ddof=1)) if diff.std(ddof=1) > 0 else 0.0

def effect(d):
    a=abs(d)
    return 'negligible' if a<0.2 else ('small' if a<0.5 else ('medium' if a<0.8 else 'large'))

for protocol, df in [('standard', per_user_std), ('long-tail', per_user_lt)]:
    cf_scores = get_scores(df, 'CF')
    for variant in ['Hybrid-Adaptive', 'Hybrid-Fixed0.5', 'Hybrid-RRF']:
        hy_scores = get_scores(df, variant)
        delta  = float((hy_scores - cf_scores).mean())
        pval   = paired_bootstrap_test(hy_scores, cf_scores, n_resamples=10_000, seed=42)
        d      = cohens_d(hy_scores, cf_scores)
        ht_rows.append({'protocol':protocol,'comparison':f'{variant} vs CF',
                        'delta_mean':round(delta,5),'p_value':round(pval,4),
                        'cohens_d':round(d,4),'effect':effect(d),'sig':pval<0.05})
        print(f'{protocol:<12} {variant+" vs CF":<32} {delta:>10.5f} {pval:>8.4f} {d:>8.4f}  {effect(d)}')

pd.DataFrame(ht_rows).to_csv(RESULTS_DIR / 'warm_hybrid_ablation_tests.csv', index=False)
print('\nSaved warm_hybrid_ablation_tests.csv')


In [ ]:
MODEL_ORDER = ['CF', 'Hybrid-Adaptive', 'Hybrid-Fixed0.5', 'Hybrid-RRF']
LABELS      = ['CF', 'Hybrid\nAdaptive', 'Hybrid\nFixed α=0.5', 'Hybrid\nRRF']
COLOR       = ['#4C72B0', '#C44E52', '#DD8452', '#55A868']
x           = np.arange(len(MODEL_ORDER))
width       = 0.35

def get_bar(agg, model, metric='ndcg', k=10):
    r = agg[(agg['model']==model)&(agg['k']==k)&(agg['metric']==metric)].iloc[0]
    return r['mean'], r['mean']-r['ci_low'], r['ci_high']-r['mean']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, (agg, protocol, n) in zip(axes, [
    (agg_std, 'Standard', len(test_data)),
    (agg_lt,  'Long-tail (top-100 excl.)', len(test_data_lt)),
]):
    vals  = [get_bar(agg, m) for m in MODEL_ORDER]
    means = [v[0] for v in vals]
    lo    = [v[1] for v in vals]
    hi    = [v[2] for v in vals]
    ax.bar(x, means, color=COLOR, edgecolor='white', yerr=[lo, hi], capsize=4)
    ax.set_xticks(x); ax.set_xticklabels(LABELS, fontsize=9)
    ax.set_ylabel('NDCG@10')
    ax.set_title(f'Warm Hybrid Ablation — {protocol}\n(95% CI, n={n} users)')
    ax.set_ylim(0, max(means)*1.30)

fig.tight_layout()
fig.savefig(RESULTS_DIR / 'warm_hybrid_ablation.png', dpi=150)
plt.close(fig)
print('Saved warm_hybrid_ablation.png')
